In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [2]:
predictors = ['V1AD02a', 'V1AD02b', 'V1AD02c', 'V1AD02d', 'V1AD02e', 'V1AD02f', 'V1AD02g','V1AD02h','V1AD02i', 'V1AD02j', 'V1AD02k']
df = pd.read_csv('../../Data/V1A.csv', usecols=predictors + ['PublicID'])

In [3]:
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [4]:
df = pd.merge(df, df_outcomes, on='PublicID', how='inner')
df

,PublicID,V1AD02a,V1AD02b,V1AD02c,V1AD02d,V1AD02e,V1AD02f,V1AD02g,V1AD02h,V1AD02i,V1AD02j,V1AD02k,MH_outcome
0,00004O,1.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,1.0,1.0,1
1,00007I,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1
2,00008G,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,0
3,00015J,1.0,2.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,0
4,00016H,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7736,17349I,1.0,2.0,2.0,2.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0,1
7737,17350A,1.0,1.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1
7738,17351V,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,0
7739,17352T,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1


In [5]:
start_col = df.columns.get_loc('V1AD02a')
end_col = df.columns.get_loc('V1AD02k') + 1  # +1 because the range end is exclusive

# Sum the specified columns row-wise and create a new column with the results
df['V1AD02a_sum'] = df.iloc[:, start_col:end_col].sum(axis=1)

In [6]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    4591
1    3150
Name: count, dtype: int64

In [7]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    binary_features=predictors
)

Dropping rows with missing values because impute=False (train: 7, test: 1).
Final dataset sizes for LR (impute=False): train=6185, test=1548
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.001, 'classifier__l1_ratio': 0.25}
Best CV Score (f1): 0.5783
Model Coefficients:
bin__V1AD02a: 0.0
bin__V1AD02b: 0.0
bin__V1AD02c: 0.0
bin__V1AD02d: 0.0
bin__V1AD02e: 0.0
bin__V1AD02f: 0.0
bin__V1AD02g: 0.0
bin__V1AD02h: 0.0
bin__V1AD02i: 0.0
bin__V1AD02j: 0.0
bin__V1AD02k: 0.0
Evaluation Metrics for LR with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5930
Precision (positive class): 0.0000
Recall (positive class): 0.0000
F1 (positive class): 0.0000
Macro Precision: 0.2965
Macro Recall: 0.5000
Macro F1: 0.3723
ROC AUC: 0.5000


In [8]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    binary_features=predictors
)

Dropping rows with missing values because impute=False (train: 7, test: 1).
Final dataset sizes for RF (impute=False): train=6185, test=1548
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best CV Score (f1): 0.5428
Feature Importances:
bin__V1AD02a: 0.1406684141726204
bin__V1AD02b: 0.08674544047002614
bin__V1AD02c: 0.06518858700076609
bin__V1AD02d: 0.06967952595928177
bin__V1AD02e: 0.07235363330912736
bin__V1AD02f: 0.07793682392264834
bin__V1AD02g: 0.27975989034105636
bin__V1AD02h: 0.08422618900344908
bin__V1AD02i: 0.04353143730268341
bin__V1AD02j: 0.044516479862991336
bin__V1AD02k: 0.03539357865534979
Evaluation Metrics for RF with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5543
Precision (positive class): 0.4653
Recall (positive class): 0.6381
F1 (positive class): 0.5382
Macro Precision: 0.5660
Macro

In [9]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    binary_features=predictors
)

Dropping rows with missing values because impute=False (train: 7, test: 1).
Final dataset sizes for XGB (impute=False): train=6185, test=1548
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.001, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
Best CV Score (f1): 0.5497
Feature Importances:
bin__V1AD02a: 0.2335169017314911
bin__V1AD02b: 0.03715607151389122
bin__V1AD02c: 0.030618775635957718
bin__V1AD02d: 0.05064383149147034
bin__V1AD02e: 0.03490935266017914
bin__V1AD02f: 0.0928458720445633
bin__V1AD02g: 0.34262585639953613
bin__V1AD02h: 0.01762143149971962
bin__V1AD02i: 0.04780620336532593
bin__V1AD02j: 0.08374041318893433
bin__V1AD02k: 0.028515242040157318
Evaluation Metrics for XGB with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5736
Precision (positive class): 0.4828
Recall (positive class): 0.6667
F1 (positive class): 0

In [10]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    binary_features=predictors
)

Dropping rows with missing values because impute=False (train: 7, test: 1).
Final dataset sizes for SVM (impute=False): train=6185, test=1548
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__kernel': 'linear'}
Best CV Score (f1): 0.5617
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and adaptive CV scoring:
Accuracy: 0.5627
Precision (positive class): 0.4758
Recall (positive class): 0.7349
F1 (positive class): 0.5777
Macro Precision: 0.5927
Macro Recall: 0.5897
Macro F1: 0.5621
ROC AUC: 0.6015


In [11]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc
# import shap

# # Assuming df is your dataframe after initial loading and merging

# # Handling missing values
# # df.fillna(df.median(), inplace=True)  # or df.mean()

# # Splitting data into features and target
# y = merged_data['MH_outcome']
# X = merged_data.drop(['PublicID', 'MH_outcome'], axis=1)

# # Splitting data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Feature scaling
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Logistic Regression Model
# model = LogisticRegression(solver='liblinear')
# model.fit(X_train_scaled, y_train)

# # Cross-Validation
# scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
# print("Cross-validated scores:", scores)

# # Hyperparameter Tuning
# param_grid = {'C': [0.1, 1, 10], 'penalty': ['l1', 'l2']}
# grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5)
# grid_search.fit(X_train_scaled, y_train)
# print("Best parameters:", grid_search.best_params_)

# # Model Evaluation
# y_pred = model.predict(X_test_scaled)
# accuracy = accuracy_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred)
# recall = recall_score(y_test, y_pred)
# f1 = f1_score(y_test, y_pred)
# print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")

# # ROC-AUC and Precision-Recall Curves
# y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
# fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
# roc_auc = auc(fpr, tpr)
# precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
# pr_auc = auc(recall, precision)

# # Plotting ROC and Precision-Recall Curves
# plt.figure(figsize=(12, 6))
# plt.subplot(1, 2, 1)
# plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend()

# plt.subplot(1, 2, 2)
# plt.plot(recall, precision, label=f'PR curve (area = {pr_auc:.2f})')
# plt.xlabel('Recall')
# plt.ylabel('Precision')
# plt.title('Precision-Recall Curve')
# plt.legend()
# plt.show()

# # Model Interpretability with SHAP
# explainer = shap.Explainer(model, X_train_scaled)
# shap_values = explainer(X_test_scaled)
# shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns)

# # Basic EDA
# # Print descriptive statistics
# print(df.describe())

# # Pairplot (this may be time-consuming for large datasets)
# # sns.pairplot(df)

# # Correlation matrix
# plt.figure(figsize=(10, 8))
# sns.heatmap(df.corr(), annot=True)
